# Information Theory: A Complete Reference Notebook
### From Shannon's Foundations to Modern AI Applications

---

> *"The fundamental problem of communication is that of reproducing at one point either exactly or approximately a message selected at another point."*  
> — Claude Shannon, *A Mathematical Theory of Communication*, 1948

---

## Table of Contents

1. [Historical Background & Motivation](#1-historical-background)
2. [What Is Information? Intuition First](#2-what-is-information)
3. [Self-Information (Surprisal)](#3-self-information)
4. [Shannon Entropy](#4-shannon-entropy)
5. [Joint, Conditional & Chain-Rule Entropy](#5-joint-and-conditional-entropy)
6. [Mutual Information](#6-mutual-information)
7. [Kullback–Leibler Divergence](#7-kl-divergence)
8. [Cross-Entropy](#8-cross-entropy)
9. [Jensen-Shannon Divergence](#9-js-divergence)
10. [Data Compression & Source Coding](#10-data-compression)
11. [Channel Capacity & Noisy Channel Theorem](#11-channel-capacity)
12. [Information Theory in Machine Learning](#12-information-in-ml)
13. [Information Theory in Deep Learning](#13-information-in-deep-learning)
14. [Information Bottleneck Theory](#14-information-bottleneck)
15. [Minimum Description Length & Bayesian Inference](#15-mdl)
16. [Further Reading & Resources](#16-further-reading)

In [ ]:
# ============================================================
# SETUP: Install and import all required packages
# ============================================================
# Uncomment the line below if running for the first time:
# !pip install numpy scipy matplotlib seaborn scikit-learn pandas

import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pandas as pd
from collections import Counter
import heapq
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})
sns.set_palette('muted')

print('All packages loaded successfully!')

---
## 1. Historical Background

### 1.1 The Pre-Shannon World

Before 1948, communication engineers designed systems largely by intuition and trial-and-error. There was no mathematical framework to answer questions like:
- What is the *minimum* number of bits needed to represent a message?
- What is the *maximum* rate at which information can be reliably sent over a noisy channel?
- Is there a fundamental, insurmountable limit to compression?

The intellectual seeds were planted much earlier:

| Year | Person | Contribution |
|------|--------|--------------|
| 1854 | George Boole | Boolean algebra — the logical foundation of binary computing |
| 1875 | Ludwig Boltzmann | Thermodynamic entropy: `S = k log W` |
| 1928 | Ralph Hartley | Logarithmic measure of information content |
| 1927–1940 | Norbert Wiener | Signal processing, cybernetics |
| **1948** | **Claude Shannon** | **A Mathematical Theory of Communication** |

### 1.2 Shannon's 1948 Revolution

Claude Shannon, working at Bell Laboratories, published the paper that founded modern information theory. His key breakthroughs:

1. **Information can be measured independently of meaning.** Whether a message says "the stock market crashed" or "the stock market rose", what matters is *how surprising* the message was — not what it meant.

2. **Entropy quantifies information.** Shannon defined the entropy of a probability distribution as a precise mathematical quantity.

3. **Source Coding Theorem:** There is a *fundamental lower bound* on how much any message can be compressed. You cannot compress below the entropy.

4. **Channel Capacity Theorem:** Every noisy channel has a maximum rate `C` at which information can be transmitted *with arbitrarily small error*. This rate is achievable (with clever encoding).

### 1.3 Why This Matters for AI

Information theory is not just telecommunications history. It is woven into the mathematical fabric of modern AI:
- **Loss functions** (cross-entropy loss in neural networks)
- **Decision trees** (information gain / entropy splitting)
- **Variational Autoencoders** (KL divergence in the ELBO objective)
- **Transformers & LLMs** (perplexity, cross-entropy language modeling)
- **Representation learning** (Information Bottleneck theory)
- **Bayesian inference** (Minimum Description Length)
- **Reinforcement learning** (entropy regularization, maximum entropy RL)

---
## 2. What Is Information? Intuition First

Before equations, let's build the right intuition.

**Core principle:** *Information is the reduction of uncertainty.*

Consider two messages:
- "The sun rose in the east today."
- "A meteorite hit downtown Tokyo today."

The first message carries almost *no* information — you already knew it with near-certainty. The second carries *a lot* of information — it is highly surprising.

**Key insight:** Information is inversely related to probability. The *less likely* an event is, the *more information* its occurrence carries.

This leads us to a key desideratum: our information measure `I(x)` for an event `x` with probability `p(x)` should:

1. Be a function of probability only: `I = f(p)`
2. Be 0 for certain events: `f(1) = 0` (no surprise = no info)
3. Increase as probability decreases: `f` is decreasing
4. Be additive for independent events: `f(p₁ · p₂) = f(p₁) + f(p₂)`

The **only** function satisfying all four properties is the logarithm:

$$I(x) = -\log_b\, p(x)$$

When `b = 2`, the unit is **bits**. When `b = e`, the unit is **nats**. When `b = 10`, it's **hartleys** (rare).

AI convention: use base `e` (nats) or base `2` (bits). PyTorch/TensorFlow use natural log (nats) internally.

---
## 3. Self-Information (Surprisal)

### Definition

The **self-information** (also called *surprisal*) of an event `x` with probability `p(x)` is:

$$\boxed{I(x) = -\log_2 p(x) \quad \text{(in bits)}}$$

### Properties

| Probability `p` | Surprisal `I = -log₂(p)` | Interpretation |
|-----------------|---------------------------|----------------|
| 1.0 (certain) | 0 bits | No surprise |
| 0.5 | 1 bit | One fair coin flip |
| 0.25 | 2 bits | Two fair coin flips |
| 0.125 | 3 bits | Three fair coin flips |
| 0.001 | ~10 bits | Very rare event |

In [ ]:
# ============================================================
# SECTION 3: Self-Information (Surprisal)
# ============================================================

def self_information(p, base=2):
    """Compute self-information (surprisal) of event with probability p.
    
    Args:
        p: probability of the event (0 < p <= 1)
        base: logarithm base (2 = bits, e = nats, 10 = hartleys)
    Returns:
        Self-information in the chosen unit
    """
    assert 0 < p <= 1, "Probability must be in (0, 1]"
    return -np.log2(p) if base == 2 else -np.log(p)


# --- Plot surprisal as a function of probability ---
p_values = np.linspace(0.001, 1.0, 1000)
surprisal = -np.log2(p_values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Surprisal curve
ax = axes[0]
ax.plot(p_values, surprisal, color='steelblue', linewidth=2.5)
ax.scatter([0.5, 0.25, 0.125, 0.0625],
           [1, 2, 3, 4],
           color='coral', zorder=5, s=80)
ax.axhline(0, color='gray', linestyle='--', alpha=0.4)
for p, bits, label in [(0.5,1,'p=0.5 → 1 bit'), (0.25,2,'p=0.25 → 2 bits'),
                        (0.125,3,'p=0.125 → 3 bits')]:
    ax.annotate(label, xy=(p, bits), xytext=(p+0.05, bits+0.3),
                fontsize=9, color='coral')
ax.set_xlabel('Probability p(x)')
ax.set_ylabel('Self-information I(x) [bits]')
ax.set_title('Surprisal: the less likely, the more informative')

# Right: Concrete example — die outcomes
ax = axes[1]
outcomes = ['Die=1', 'Die=2', 'Die=3', 'Die=4', 'Die=5', 'Die=6',
            'Coin=H', 'Coin=T', 'Rare(1%)','Very rare\n(0.1%)']
probs    = [1/6]*6 + [1/2, 1/2, 0.01, 0.001]
info     = [-np.log2(p) for p in probs]
colors   = ['steelblue']*6 + ['seagreen','seagreen','coral','crimson']
bars = ax.barh(outcomes, info, color=colors, alpha=0.8)
for bar, val in zip(bars, info):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.2f} bits', va='center', fontsize=9)
ax.set_xlabel('Self-information [bits]')
ax.set_title('Surprisal of different events')

plt.tight_layout()
plt.suptitle('Section 3: Self-Information (Surprisal)', y=1.02, fontsize=14, fontweight='bold')
plt.show()

# Demonstrate additivity for independent events
print("=== Additivity of Information ===")
p_heads = 0.5
p_two_heads = p_heads * p_heads  # Independent coin flips
print(f"P(H)  = {p_heads}  → I = {self_information(p_heads):.4f} bits")
print(f"P(HH) = {p_two_heads} → I = {self_information(p_two_heads):.4f} bits")
print(f"I(H) + I(H) = {2 * self_information(p_heads):.4f} bits  ✓ (equal, as expected)")

---
## 4. Shannon Entropy

### Motivation

Self-information tells us about *one specific outcome*. But in practice, we want to characterize the *entire distribution* — before we know the outcome. How surprising is a source *on average*?

### Definition

The **Shannon entropy** `H(X)` of a discrete random variable `X` with distribution `p` is the *expected surprisal*:

$$\boxed{H(X) = -\sum_{x \in \mathcal{X}} p(x) \log_2 p(x) = \mathbb{E}_{x \sim p}[-\log_2 p(x)]}$$

Convention: `0 · log(0) = 0` (by continuity: `lim_{p→0} p log p = 0`).

### Interpretation

- `H(X)` is the **average number of bits** needed to describe outcomes of `X`.
- It measures the **uncertainty** or **unpredictability** of `X`.
- It is the **fundamental lower bound** on lossless compression (Shannon's Source Coding Theorem).

### Properties of Entropy

| Property | Formula | Meaning |
|----------|---------|----------|
| Non-negativity | `H(X) ≥ 0` | Entropy is never negative |
| Maximum at uniformity | `H(X) ≤ log₂|X|` | Uniform distribution maximizes entropy |
| Zero for certainty | `H(X) = 0` iff deterministic | No uncertainty = no information |
| Concavity | `H(λp + (1-λ)q) ≥ λH(p) + (1-λ)H(q)` | Mixing increases entropy |
| Additivity (independent) | `H(X,Y) = H(X) + H(Y)` if independent | |

### Continuous Entropy (Differential Entropy)

For continuous distributions, the analogous quantity is **differential entropy**:

$$h(X) = -\int_{-\infty}^{\infty} p(x) \ln p(x)\, dx$$

Note: differential entropy *can* be negative (unlike discrete entropy).

In [ ]:
# ============================================================
# SECTION 4: Shannon Entropy
# ============================================================

def entropy(probs, base=2):
    """Compute Shannon entropy of a discrete distribution.
    
    Args:
        probs: array of probabilities (must sum to 1)
        base:  log base (2=bits, np.e=nats)
    Returns:
        Entropy value
    """
    probs = np.array(probs, dtype=float)
    assert np.isclose(probs.sum(), 1.0), f"Probabilities must sum to 1, got {probs.sum()}"
    # 0 * log(0) = 0 by convention
    mask = probs > 0
    if base == 2:
        return -np.sum(probs[mask] * np.log2(probs[mask]))
    else:
        return -np.sum(probs[mask] * np.log(probs[mask])) / np.log(base)


# --- Example 1: Entropy of a biased coin ---
print("=== Entropy of a biased coin ===")
for p_heads in [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]:
    if p_heads in [0.0, 1.0]:
        h = 0.0
    else:
        h = entropy([p_heads, 1-p_heads])
    print(f"  p(H) = {p_heads:.2f} → H = {h:.4f} bits")

print()

# --- Example 2: Entropy of a 6-sided die ---
fair_die = [1/6] * 6
loaded_die = [0.5, 0.1, 0.1, 0.1, 0.1, 0.1]  # biased toward 1
print(f"Fair die entropy:   {entropy(fair_die):.4f} bits (max = log2(6) = {np.log2(6):.4f})")
print(f"Loaded die entropy: {entropy(loaded_die):.4f} bits (lower: more predictable)")

print()

# --- Plot: Binary entropy function ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Binary entropy H(p)
ax = axes[0]
p = np.linspace(0.001, 0.999, 1000)
H = -p * np.log2(p) - (1-p) * np.log2(1-p)
ax.plot(p, H, 'steelblue', linewidth=2.5)
ax.fill_between(p, H, alpha=0.1, color='steelblue')
ax.scatter([0.5], [1.0], color='coral', s=100, zorder=5, label='Max at p=0.5: H=1 bit')
ax.set_xlabel('p (probability of heads)')
ax.set_ylabel('H(p) [bits]')
ax.set_title('Binary entropy function')
ax.legend(fontsize=9)

# Plot 2: Entropy vs number of uniform outcomes
ax = axes[1]
n_vals = np.arange(1, 33)
H_uniform = np.log2(n_vals)
ax.plot(n_vals, H_uniform, 'o-', color='seagreen', linewidth=2)
for n in [2, 4, 8, 16, 32]:
    ax.annotate(f'n={n}\n{np.log2(n):.1f}b',
                xy=(n, np.log2(n)), xytext=(n+0.5, np.log2(n)-0.3),
                fontsize=8, color='seagreen')
ax.set_xlabel('Number of equally-likely outcomes')
ax.set_ylabel('H [bits]')
ax.set_title('Entropy of uniform distribution')

# Plot 3: Comparing distributions visually
ax = axes[2]
distributions = {
    'Certain\n(degenerate)':  [1.0, 0.0, 0.0, 0.0],
    'Skewed':    [0.7, 0.1, 0.1, 0.1],
    'Moderate':  [0.4, 0.3, 0.2, 0.1],
    'Uniform':   [0.25]*4,
}
x = np.arange(4)
offsets = np.linspace(-0.3, 0.3, 4)
colors_list = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71']
for (name, dist), offset, col in zip(distributions.items(), offsets, colors_list):
    h = entropy([p for p in dist if p > 0] + [1e-10]*(4-sum(1 for p in dist if p>0)))
    h_real = entropy(dist) if all(p >= 0 for p in dist) and abs(sum(dist)-1)<1e-9 else 0
    ax.bar(x + offset, dist, width=0.2, label=f'{name}\nH={h_real:.2f}b', color=col, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(['A', 'B', 'C', 'D'])
ax.set_ylabel('Probability')
ax.set_title('Higher uncertainty = higher entropy')
ax.legend(fontsize=8, loc='upper right')

plt.suptitle('Section 4: Shannon Entropy', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Joint, Conditional & Chain-Rule Entropy

### 5.1 Joint Entropy

For two random variables `X` and `Y` with joint distribution `p(x, y)`:

$$H(X, Y) = -\sum_{x,y} p(x,y) \log_2 p(x,y)$$

**Intuition:** The total uncertainty in the pair `(X, Y)` together.

### 5.2 Conditional Entropy

The entropy of `Y` *given that we know* `X`:

$$H(Y | X) = -\sum_{x,y} p(x,y) \log_2 p(y|x) = \mathbb{E}_x[H(Y|X=x)]$$

**Intuition:** The remaining uncertainty about `Y` after observing `X`.

Key property: `H(Y|X) ≤ H(Y)` — knowing `X` cannot *increase* uncertainty about `Y`.

### 5.3 Chain Rule of Entropy

$$H(X, Y) = H(X) + H(Y|X) = H(Y) + H(X|Y)$$

Generalization to multiple variables:

$$H(X_1, X_2, \ldots, X_n) = \sum_{i=1}^{n} H(X_i | X_1, \ldots, X_{i-1})$$

**Why this matters for LLMs:** The cross-entropy loss in a language model is exactly the chain rule entropy! Predicting token $t_i$ given all previous tokens $t_1, \ldots, t_{i-1}$ is computing $H(T_i | T_1, \ldots, T_{i-1})$.

### 5.4 The Entropy Diagram

The relationships are elegantly captured by the **information diagram**:

```
          H(X,Y)
         /       \
       H(X)     H(Y)
         \  I(X;Y) /
     H(X|Y)     H(Y|X)
```

$$H(X, Y) = H(X) + H(Y) - I(X;Y)$$

In [ ]:
# ============================================================
# SECTION 5: Joint, Conditional Entropy and Chain Rule
# ============================================================

def joint_entropy(joint_probs):
    """Entropy of a joint distribution p(x,y).
    joint_probs: 2D numpy array where entry [i,j] = p(x_i, y_j)
    """
    flat = joint_probs.flatten()
    mask = flat > 0
    return -np.sum(flat[mask] * np.log2(flat[mask]))

def conditional_entropy_YgivenX(joint_probs):
    """H(Y|X) from joint distribution.
    H(Y|X) = H(X,Y) - H(X)
    """
    marginal_x = joint_probs.sum(axis=1)  # p(x)
    hx = entropy(marginal_x)
    hxy = joint_entropy(joint_probs)
    return hxy - hx


# --- Concrete example: Weather and mood ---
# X = Weather (Sunny, Cloudy)
# Y = Mood    (Happy, Sad)
# joint_probs[weather, mood]
print("=== Example: Weather (X) and Mood (Y) ===")

# Correlated case: sunny → happy, cloudy → sad
joint_corr = np.array([
    [0.40, 0.10],   # Sunny: mostly happy
    [0.05, 0.45],   # Cloudy: mostly sad
])
# p_X = [0.5, 0.5], p_Y = [0.45, 0.55]

# Independent case: same marginals but no correlation
p_x = joint_corr.sum(axis=1)
p_y = joint_corr.sum(axis=0)
joint_indep = np.outer(p_x, p_y)   # p(x,y) = p(x)p(y)

for label, J in [('Correlated', joint_corr), ('Independent', joint_indep)]:
    px = J.sum(axis=1)
    py = J.sum(axis=0)
    HX  = entropy(px)
    HY  = entropy(py)
    HXY = joint_entropy(J)
    HYgX = conditional_entropy_YgivenX(J)
    HXgY = HXY - HY
    IMI  = HX + HY - HXY  # Mutual information
    print(f"\n  [{label}]")
    print(f"    H(X)   = {HX:.4f} bits")
    print(f"    H(Y)   = {HY:.4f} bits")
    print(f"    H(X,Y) = {HXY:.4f} bits")
    print(f"    H(Y|X) = {HYgX:.4f} bits  [uncertainty about mood given weather]")
    print(f"    H(X|Y) = {HXgY:.4f} bits")
    print(f"    I(X;Y) = {IMI:.4f} bits  [mutual information]")
    print(f"    Chain rule check: H(X)+H(Y|X) = {HX + HYgX:.4f} ≈ H(X,Y) = {HXY:.4f}")

# --- Visualize the Venn diagram of information quantities ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, J) in zip(axes, [('Correlated (high MI)', joint_corr), 
                                    ('Independent (zero MI)', joint_indep)]):
    px = J.sum(axis=1)
    py = J.sum(axis=0)
    HX  = entropy(px)
    HY  = entropy(py)
    HXY = joint_entropy(J)
    MI  = HX + HY - HXY

    from matplotlib.patches import Circle
    overlap = MI / max(HX, HY)  # normalized overlap for display
    sep = 0.8 - overlap * 0.6

    c1 = Circle((-sep/2, 0), 0.7, color='steelblue', alpha=0.4, label=f'H(X)={HX:.2f}b')
    c2 = Circle(( sep/2, 0), 0.7, color='coral',    alpha=0.4, label=f'H(Y)={HY:.2f}b')
    ax.add_patch(c1)
    ax.add_patch(c2)
    ax.text(-sep/2 - 0.35, 0, f'H(X|Y)\n{HX-MI:.2f}b', ha='center', va='center', fontsize=9)
    ax.text( sep/2 + 0.35, 0, f'H(Y|X)\n{HY-MI:.2f}b', ha='center', va='center', fontsize=9)
    ax.text(0, 0, f'I(X;Y)\n{MI:.2f}b', ha='center', va='center', fontsize=9, fontweight='bold')
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f'{label}\nH(X,Y)={HXY:.2f}b')
    ax.legend(loc='lower center', fontsize=9)

plt.suptitle('Section 5: Information Venn Diagrams', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Mutual Information

### Definition

**Mutual information** `I(X;Y)` measures the amount of information that `X` and `Y` share — how much knowing one variable reduces uncertainty about the other:

$$\boxed{I(X;Y) = H(X) - H(X|Y) = H(Y) - H(Y|X) = H(X) + H(Y) - H(X,Y)}$$

Equivalently, using the KL divergence:

$$I(X;Y) = D_{\text{KL}}(p(x,y) \| p(x)p(y))$$

This shows that MI measures how far the joint distribution is from the product of marginals — i.e., how *non-independent* X and Y are.

### Properties

| Property | Statement |
|----------|----------|
| Non-negativity | `I(X;Y) ≥ 0`, with equality iff X,Y independent |
| Symmetry | `I(X;Y) = I(Y;X)` |
| Upper bound | `I(X;Y) ≤ min(H(X), H(Y))` |
| Data processing inequality | If `X → Y → Z` is a Markov chain, then `I(X;Z) ≤ I(X;Y)` |

### AI Applications of Mutual Information

- **Feature selection:** Select features with highest MI with the target variable
- **Decision tree splitting (information gain):** Choose splits that maximize MI between feature and label
- **Independent Component Analysis (ICA):** Find components that minimize MI
- **Information Bottleneck:** Compress representations while maximizing MI with labels
- **Mutual Information Neural Estimation (MINE):** Deep learning approach to estimate MI

In [ ]:
# ============================================================
# SECTION 6: Mutual Information
# ============================================================

from sklearn.feature_selection import mutual_info_classif
from sklearn.datasets import load_iris

def mutual_information(joint_probs):
    """I(X;Y) = H(X) + H(Y) - H(X,Y)"""
    px  = joint_probs.sum(axis=1)
    py  = joint_probs.sum(axis=0)
    HX  = entropy(px)
    HY  = entropy(py)
    HXY = joint_entropy(joint_probs)
    return HX + HY - HXY


# --- Demonstrate MI across a range of correlations ---
print("=== Mutual Information across correlation strengths ===")
rho_vals = np.linspace(-0.99, 0.99, 200)
mi_vals  = []

for rho in rho_vals:
    # Bivariate Normal: MI = -0.5 * log(1 - rho^2)
    # (analytical result for Gaussian)
    mi = -0.5 * np.log2(1 - rho**2)
    mi_vals.append(mi)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: MI vs correlation for bivariate normal
ax = axes[0]
ax.plot(rho_vals, mi_vals, 'steelblue', linewidth=2.5)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Correlation ρ')
ax.set_ylabel('I(X;Y) [bits]')
ax.set_title('MI of bivariate Normal\n(analytical: −½ log₂(1−ρ²))')

# Center: scatter plots at different correlations
ax = axes[1]
np.random.seed(42)
for rho, color, label in [(-0.9, 'coral', 'ρ=−0.9'), (0.0, 'gray', 'ρ=0'), (0.9, 'steelblue', 'ρ=0.9')]:
    cov = [[1, rho], [rho, 1]]
    X_data = np.random.multivariate_normal([0,0], cov, 300)
    mi_val = -0.5 * np.log2(1 - rho**2)
    ax.scatter(X_data[:,0], X_data[:,1], alpha=0.3, color=color, s=10,
               label=f'{label}, I={mi_val:.2f}b')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Scatter at different correlations')
ax.legend(fontsize=9)

# Right: Mutual information for Iris features (sklearn)
ax = axes[2]
iris = load_iris()
mi_scores = mutual_info_classif(iris.data, iris.target, discrete_features=False, random_state=42)
feature_names = ['Sepal\nlength', 'Sepal\nwidth', 'Petal\nlength', 'Petal\nwidth']
bars = ax.bar(feature_names, mi_scores, color=['steelblue','gray','coral','seagreen'])
for bar, val in zip(bars, mi_scores):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.3f}',
            ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Mutual Information [nats]')
ax.set_title('MI between Iris features\nand class label')

plt.suptitle('Section 6: Mutual Information', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nIris MI scores (higher = more predictive of class):")
for name, score in zip(iris.feature_names, mi_scores):
    print(f"  {name}: {score:.4f} nats")

---
## 7. Kullback–Leibler (KL) Divergence

### Definition

The **KL divergence** (also called *relative entropy*) from distribution `Q` to distribution `P` measures how much information is *lost* when we use `Q` to approximate `P`:

$$\boxed{D_{\text{KL}}(P \| Q) = \sum_x p(x) \log_2 \frac{p(x)}{q(x)} = \mathbb{E}_{x \sim P}\left[\log_2 \frac{p(x)}{q(x)}\right]}$$

For continuous distributions:
$$D_{\text{KL}}(P \| Q) = \int p(x) \ln \frac{p(x)}{q(x)}\, dx$$

### Critical Properties

| Property | Detail |
|----------|--------|
| Non-negative | `D_KL(P‖Q) ≥ 0`, zero iff `P = Q` (Gibbs' inequality) |
| **Asymmetric** | `D_KL(P‖Q) ≠ D_KL(Q‖P)` in general — *not* a metric |
| Infinite if support issue | If `Q(x) = 0` but `P(x) > 0`, then `D_KL = ∞` |
| Connection to cross-entropy | `D_KL(P‖Q) = H(P, Q) - H(P)` |

### Asymmetry Explained

- **Forward KL** `D_KL(P‖Q)`: Penalizes Q being small where P is large. `Q` must cover all of `P`. → *mass-covering* behavior. Used in maximum likelihood.
- **Reverse KL** `D_KL(Q‖P)`: Penalizes Q being large where P is small. → *mode-seeking* behavior. Used in variational inference.

### AI Applications

- **VAEs:** `D_KL(q(z|x) ‖ p(z))` in the ELBO objective
- **Policy gradient (PPO):** KL constraint between old and new policy
- **Knowledge distillation:** KL between teacher and student softmax outputs
- **RLHF/DPO:** KL penalty to keep fine-tuned model close to reference

In [ ]:
# ============================================================
# SECTION 7: KL Divergence
# ============================================================

def kl_divergence(P, Q, base=2):
    """D_KL(P || Q) — how much Q approximates P poorly.
    
    CAUTION: returns np.inf if Q(x)=0 where P(x)>0
    """
    P, Q = np.array(P, dtype=float), np.array(Q, dtype=float)
    mask = P > 0
    if np.any(Q[mask] == 0):
        return np.inf
    if base == 2:
        return np.sum(P[mask] * np.log2(P[mask] / Q[mask]))
    else:
        return np.sum(P[mask] * np.log(P[mask] / Q[mask]))


# --- Gaussian KL demonstration ---
x = np.linspace(-6, 6, 1000)

def gaussian_kl_analytical(mu1, sigma1, mu2, sigma2):
    """Analytical KL divergence between two Gaussians (in nats).
    D_KL(N(mu1,s1) || N(mu2,s2)) = log(s2/s1) + (s1^2 + (mu1-mu2)^2)/(2*s2^2) - 0.5
    """
    return (np.log(sigma2/sigma1) 
            + (sigma1**2 + (mu1-mu2)**2) / (2*sigma2**2) 
            - 0.5)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: Asymmetry demo
ax = axes[0]
# P = narrow, Q = broad
P_dist = stats.norm(0, 1).pdf(x)
Q_dist = stats.norm(0, 2).pdf(x)
P_dist /= P_dist.sum(); Q_dist /= Q_dist.sum()
kl_pq = kl_divergence(P_dist, Q_dist)
kl_qp = kl_divergence(Q_dist, P_dist)

ax.plot(x, P_dist * len(x)/12, label='P = N(0,1)', color='steelblue', linewidth=2)
ax.plot(x, Q_dist * len(x)/12, label='Q = N(0,2)', color='coral', linewidth=2, linestyle='--')
ax.set_title(f'KL Asymmetry\nD_KL(P‖Q)={kl_pq:.3f}b,  D_KL(Q‖P)={kl_qp:.3f}b')
ax.set_xlabel('x')
ax.set_ylabel('Density (scaled)')
ax.legend()

# Center: KL as function of mean shift
ax = axes[1]
mean_shifts = np.linspace(0, 4, 100)
kl_vals = [gaussian_kl_analytical(0, 1, mu, 1) for mu in mean_shifts]
kl_sigma = [gaussian_kl_analytical(0, 1, 0, s) for s in np.linspace(0.5, 4, 100)]

ax.plot(mean_shifts, kl_vals, 'steelblue', linewidth=2, label='Shift mean (σ=1 fixed)')
ax.plot(np.linspace(0.5, 4, 100) - 0.5, kl_sigma, 'coral',
        linewidth=2, linestyle='--', label='Scale σ (μ=0 fixed)')
ax.set_xlabel('Parameter change')
ax.set_ylabel('D_KL (nats)')
ax.set_title('KL grows quadratically\nwith mean shift')
ax.legend(fontsize=9)

# Right: Forward vs Reverse KL mode-covering vs mode-seeking
ax = axes[2]
# P = bimodal
P_dist2 = stats.norm(-2, 0.7).pdf(x) * 0.5 + stats.norm(2, 0.7).pdf(x) * 0.5
# Forward KL approximation: broad Gaussian covering both modes
Q_forward = stats.norm(0, 2.2).pdf(x)
# Reverse KL approximation: narrow Gaussian at one mode
Q_reverse = stats.norm(2, 0.7).pdf(x)

ax.fill_between(x, P_dist2/P_dist2.max(), alpha=0.3, color='steelblue', label='P (bimodal, true)')
ax.plot(x, Q_forward/Q_forward.max(), 'coral', linewidth=2,
        linestyle='--', label='Forward KL: mass-covering')
ax.plot(x, Q_reverse/Q_reverse.max(), 'seagreen', linewidth=2,
        label='Reverse KL: mode-seeking')
ax.set_xlabel('x')
ax.set_title('Forward vs Reverse KL\nbehavior on bimodal target')
ax.legend(fontsize=9)

plt.suptitle('Section 7: KL Divergence', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Concrete numbers
print("KL divergences between Gaussians (analytical, in nats):")
for (mu1,s1,mu2,s2) in [(0,1,0,1), (0,1,1,1), (0,1,0,2), (0,1,2,3)]:
    kl = gaussian_kl_analytical(mu1,s1,mu2,s2)
    print(f"  D_KL(N({mu1},{s1}) ‖ N({mu2},{s2})) = {kl:.4f} nats")

---
## 8. Cross-Entropy

### Definition

The **cross-entropy** of distribution `Q` relative to `P` is:

$$\boxed{H(P, Q) = -\sum_x p(x) \log_2 q(x) = H(P) + D_{\text{KL}}(P \| Q)}$$

**Intuition:** Cross-entropy is the average number of bits needed to encode events drawn from `P`, using a code designed for `Q`. If `Q = P`, this equals the entropy `H(P)` (the minimum). Any mismatch adds extra bits equal to `D_KL(P‖Q)`.

### Cross-Entropy as a Loss Function

In classification, we want our model's predicted distribution `Q = model(x)` to match the true label distribution `P = one-hot(y)`. Minimizing cross-entropy loss is equivalent to:

1. **Minimizing KL divergence** `D_KL(P‖Q)` (since `H(P)` is a constant w.r.t. model parameters)
2. **Maximum Likelihood Estimation** (MLE) — mathematically equivalent

For binary classification with true label `y ∈ {0,1}` and predicted probability `ŷ = σ(z)`:

$$\mathcal{L}_{\text{BCE}} = -y \log \hat{y} - (1-y) \log(1-\hat{y})$$

For multi-class with one-hot label `y` and softmax output `ŷ`:

$$\mathcal{L}_{\text{CE}} = -\sum_k y_k \log \hat{y}_k$$

### Connection to Language Models

In a language model, we train to minimize:

$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log P_{\text{model}}(w_t | w_1, \ldots, w_{t-1})$$

This is the cross-entropy of the model distribution relative to the empirical data distribution. The exponentiated form is **perplexity**: `PPL = 2^H(P,Q)`, where lower is better.

In [ ]:
# ============================================================
# SECTION 8: Cross-Entropy and Loss Functions
# ============================================================

def cross_entropy(P, Q, base=2):
    """H(P, Q) = -sum_x P(x) log Q(x)"""
    P, Q = np.array(P, dtype=float), np.array(Q, dtype=float)
    mask = P > 0
    if base == 2:
        return -np.sum(P[mask] * np.log2(Q[mask]))
    return -np.sum(P[mask] * np.log(Q[mask]))

def perplexity(P, Q):
    """Perplexity = 2^H(P,Q) for language model evaluation."""
    return 2 ** cross_entropy(P, Q, base=2)


# --- Demo 1: Cross-entropy loss vs predicted probability ---
print("=== Cross-entropy loss (binary classification, true label = 1) ===")
y_hat_vals = np.linspace(0.01, 0.99, 100)
loss_correct   = -np.log2(y_hat_vals)       # true label = 1
loss_incorrect = -np.log2(1 - y_hat_vals)   # true label = 0

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.plot(y_hat_vals, loss_correct,   'steelblue', linewidth=2.5, label='True label = 1')
ax.plot(y_hat_vals, loss_incorrect, 'coral', linewidth=2.5, linestyle='--', label='True label = 0')
ax.axvline(0.5, color='gray', alpha=0.3, linestyle=':')
ax.set_xlabel('Predicted probability ŷ')
ax.set_ylabel('Binary cross-entropy loss [bits]')
ax.set_title('Loss goes to ∞ as prediction\napproaches wrong extreme')
ax.legend()
ax.set_ylim(0, 7)

# --- Demo 2: Cross-entropy decomposition ---
ax = axes[1]
# True distribution P (e.g., document class probabilities)
P_true = np.array([0.5, 0.3, 0.15, 0.05])
# Three model predictions
Q_perfect = P_true.copy()
Q_ok      = np.array([0.4, 0.3, 0.2, 0.1])
Q_bad     = np.array([0.1, 0.1, 0.1, 0.7])

classes = ['A', 'B', 'C', 'D']
x_pos = np.arange(4)
ax.bar(x_pos - 0.3, P_true,    width=0.2, label='True P', color='steelblue', alpha=0.9)
ax.bar(x_pos - 0.1, Q_perfect, width=0.2, label=f'Perfect Q  CE={cross_entropy(P_true,Q_perfect):.2f}b',
       color='seagreen', alpha=0.7)
ax.bar(x_pos + 0.1, Q_ok,      width=0.2, label=f'OK Q       CE={cross_entropy(P_true,Q_ok):.2f}b',
       color='orange', alpha=0.7)
ax.bar(x_pos + 0.3, Q_bad,     width=0.2, label=f'Bad Q      CE={cross_entropy(P_true,Q_bad):.2f}b',
       color='coral', alpha=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels(classes)
ax.set_ylabel('Probability')
ax.set_title('Cross-entropy increases\nwith distribution mismatch')
ax.legend(fontsize=8)

# --- Demo 3: Perplexity in language modeling ---
ax = axes[2]
# Simulate: how model perplexity correlates with CE loss
ce_losses = np.linspace(1.0, 6.0, 100)
ppl_vals  = 2 ** ce_losses
ax.plot(ce_losses, ppl_vals, 'purple', linewidth=2.5)
for ce, label in [(2, 'Simple text\nPPL~4'), (4, 'Average LM\nPPL~16'), (6, 'Poor LM\nPPL~64')]:
    ax.scatter([ce], [2**ce], color='coral', zorder=5, s=80)
    ax.annotate(label, xy=(ce, 2**ce), xytext=(ce+0.15, 2**ce * 0.7), fontsize=8)
ax.set_xlabel('Cross-entropy loss [bits/token]')
ax.set_ylabel('Perplexity (PPL = 2^CE)')
ax.set_title('Perplexity: the exponential\npenalty of CE loss')

plt.suptitle('Section 8: Cross-Entropy and Perplexity', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nDecomposition: H(P,Q) = H(P) + D_KL(P||Q)")
for name, Q in [('Perfect', Q_perfect), ('OK', Q_ok), ('Bad', Q_bad)]:
    HP   = entropy(P_true)
    kl   = kl_divergence(P_true, Q)
    ce   = cross_entropy(P_true, Q)
    print(f"  {name:8s}: H(P)={HP:.3f}  +  KL={kl:.3f}  =  CE={ce:.3f}  PPL={2**ce:.2f}")

---
## 9. Jensen-Shannon Divergence

### Motivation

KL divergence is asymmetric and can be infinite. The **Jensen-Shannon Divergence (JSD)** is a symmetric, bounded, and smooth version:

$$\boxed{D_{\text{JS}}(P \| Q) = \frac{1}{2} D_{\text{KL}}(P \| M) + \frac{1}{2} D_{\text{KL}}(Q \| M), \quad M = \frac{P+Q}{2}}$$

### Properties

- `0 ≤ JSD ≤ 1` (using log base 2)
- Symmetric: `JSD(P‖Q) = JSD(Q‖P)`
- `JSD = 0` iff `P = Q`
- `√JSD` is a *metric* (satisfies triangle inequality!)
- Defined even when supports don't overlap (unlike KL)

### AI Applications

- **Generative Adversarial Networks (GANs):** The original GAN objective (Goodfellow et al., 2014) minimizes JSD between real and generated distributions. The discriminator objective is related to JSD.
- **Text similarity:** Comparing topic distributions of documents
- **Ensemble diversity:** Measuring diversity between model predictions

In [ ]:
# ============================================================
# SECTION 9: Jensen-Shannon Divergence & GANs
# ============================================================

def js_divergence(P, Q, base=2):
    """Jensen-Shannon divergence (symmetric version of KL)."""
    P, Q = np.array(P, dtype=float), np.array(Q, dtype=float)
    M = 0.5 * (P + Q)
    return 0.5 * kl_divergence(P, M, base) + 0.5 * kl_divergence(Q, M, base)

def js_distance(P, Q):
    """sqrt(JSD) — a proper metric."""
    return np.sqrt(js_divergence(P, Q))


x = np.linspace(-8, 8, 1000)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: KL vs JSD comparison across mean separation
ax = axes[0]
mean_seps = np.linspace(0, 4, 100)
kl_vals_sep  = []
jsd_vals_sep = []
for sep in mean_seps:
    P_d = stats.norm(0, 1).pdf(x);  P_d /= P_d.sum()
    Q_d = stats.norm(sep, 1).pdf(x); Q_d /= Q_d.sum()
    Q_d = np.clip(Q_d, 1e-15, None); Q_d /= Q_d.sum()
    kl_vals_sep.append(kl_divergence(P_d, Q_d))
    jsd_vals_sep.append(js_divergence(P_d, Q_d))

ax.plot(mean_seps, kl_vals_sep,  'coral', linewidth=2, label='KL(P‖Q)')
ax.plot(mean_seps, jsd_vals_sep, 'steelblue', linewidth=2, label='JSD(P‖Q)')
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.4, label='JSD max = 1')
ax.set_xlabel('Mean separation')
ax.set_ylabel('Divergence [bits]')
ax.set_title('KL diverges; JSD saturates\nat 1 bit when fully separated')
ax.legend(fontsize=9)

# Center: GAN training intuition
ax = axes[1]
# P_real = bimodal, show how generated distribution converges
p_real = stats.norm(-2, 0.8).pdf(x)*0.5 + stats.norm(2, 0.8).pdf(x)*0.5
stages = [
    ('Epoch 0 (noise)', stats.norm(0, 3).pdf(x), 'lightcoral', '--'),
    ('Epoch 50', stats.norm(0, 1.5).pdf(x)*0.3 + stats.norm(1.5, 1).pdf(x)*0.7, 'orange', '--'),
    ('Epoch 200 (converged)', p_real * 0.98, 'seagreen', '-'),
]
ax.fill_between(x, p_real/p_real.max(), alpha=0.25, color='steelblue', label='P_real')
ax.plot(x, p_real/p_real.max(), 'steelblue', linewidth=2)
for label, dist, color, ls in stages:
    d = dist/dist.max()
    ax.plot(x, d, color=color, linewidth=1.5, linestyle=ls, label=label)
ax.set_xlim(-6, 6)
ax.set_title('GAN training: generator\nminimizes JSD with real data')
ax.legend(fontsize=8)
ax.set_xlabel('x')

# Right: Symmetric comparison table
ax = axes[2]
props = {
    'Symmetry': ['No', 'No', 'Yes'],
    'Bounded': ['No', 'No', '0–1'],
    'Metric': ['No', 'No', 'Yes (√JSD)'],
    'Works w/\nno-overlap': ['No (∞)', 'No (∞)', 'Yes'],
    'GAN connection': ['No', 'No', 'Yes'],
}
measures = ['KL(P‖Q)', 'KL(Q‖P)', 'JSD']
table_data = [[props[k][i] for i in range(3)] for k in props]
table = ax.table(
    cellText=table_data,
    rowLabels=list(props.keys()),
    colLabels=measures,
    cellLoc='center',
    loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#d6eaf8')
    elif col == 2 and row > 0:  # JSD column
        cell.set_facecolor('#d5f5e3')
ax.axis('off')
ax.set_title('Divergence comparison', pad=20)

plt.suptitle('Section 9: Jensen-Shannon Divergence', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 10. Data Compression & Source Coding

### Shannon's Source Coding Theorem

> **Theorem (Shannon, 1948):** Let a source have entropy `H` bits/symbol. Then:
> 1. It is impossible to compress the source to fewer than `H` bits/symbol on average.
> 2. It is possible to compress the source to arbitrarily close to `H` bits/symbol (using long block codes).

Entropy is the **fundamental lower bound** on lossless compression.

### Huffman Coding

**Huffman coding** is an optimal prefix-free code that assigns shorter codes to more frequent symbols. It is *optimal* in the sense that it achieves the minimum average code length `L` where:

$$H(X) \leq L < H(X) + 1$$

**Algorithm:**
1. Create a leaf node for each symbol with its frequency
2. Build a priority queue (min-heap) ordered by frequency
3. Repeatedly extract the two lowest-frequency nodes, create a parent with combined frequency
4. Assign `0` to left branches, `1` to right branches
5. Read codes from root to leaves

### Modern Compression

- **Arithmetic coding:** Approaches entropy more closely than Huffman
- **ANS (Asymmetric Numeral Systems):** Used in modern codecs (zstd, LZFSE)
- **LZ77/LZ78:** Context-based compression (ZIP, gzip)
- **Neural compression:** Using learned probabilistic models (transformers!) for near-optimal compression

In [ ]:
# ============================================================
# SECTION 10: Huffman Coding and Source Compression
# ============================================================

class HuffmanNode:
    def __init__(self, symbol, freq):
        self.symbol = symbol
        self.freq   = freq
        self.left   = None
        self.right  = None
    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(frequencies):
    """Build Huffman tree from {symbol: frequency} dict."""
    heap = [HuffmanNode(sym, freq) for sym, freq in frequencies.items()]
    heapq.heapify(heap)
    while len(heap) > 1:
        left  = heapq.heappop(heap)
        right = heapq.heappop(heap)
        parent = HuffmanNode(None, left.freq + right.freq)
        parent.left  = left
        parent.right = right
        heapq.heappush(heap, parent)
    return heap[0]

def get_codes(node, prefix='', codebook=None):
    """Extract binary codes from Huffman tree."""
    if codebook is None:
        codebook = {}
    if node.symbol is not None:
        codebook[node.symbol] = prefix if prefix else '0'
    else:
        get_codes(node.left,  prefix + '0', codebook)
        get_codes(node.right, prefix + '1', codebook)
    return codebook


# --- Apply Huffman coding to English letter frequencies ---
# Approximate frequencies of letters in English text
english_freq = {
    'E': 12.7, 'T': 9.1, 'A': 8.2, 'O': 7.5, 'I': 7.0,
    'N': 6.7, 'S': 6.3, 'H': 6.1, 'R': 6.0, 'D': 4.3,
    'L': 4.0, 'C': 2.8, 'U': 2.8, 'M': 2.4, 'W': 2.4,
    'F': 2.2, 'G': 2.0, 'Y': 2.0, 'P': 1.9, 'B': 1.5,
}

# Normalize to probabilities
total = sum(english_freq.values())
probs = {k: v/total for k, v in english_freq.items()}

# Build and decode Huffman tree
tree = build_huffman_tree(english_freq)
codes = get_codes(tree)

# Compute theoretical entropy and actual average code length
prob_list = list(probs.values())
H = entropy(prob_list)
avg_length = sum(probs[sym] * len(code) for sym, code in codes.items())
fixed_length = np.log2(len(english_freq))  # naive fixed-length encoding

print(f"Theoretical entropy:      {H:.4f} bits/symbol")
print(f"Huffman average length:   {avg_length:.4f} bits/symbol  (efficiency: {H/avg_length*100:.1f}%)")
print(f"Fixed-length (naive):     {fixed_length:.4f} bits/symbol")
print(f"Huffman savings vs fixed: {(1 - avg_length/fixed_length)*100:.1f}%")
print()
print("Huffman codes for top 10 letters:")
sorted_codes = sorted(codes.items(), key=lambda x: len(x[1]))
for sym, code in sorted_codes[:10]:
    print(f"  {sym}: {code:10s} ({len(code)} bits, p={probs[sym]:.3f})")


# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
letters = list(probs.keys())
code_lengths = [len(codes[l]) for l in letters]
p_sorted = [probs[l] for l in letters]
sorted_data = sorted(zip(p_sorted, code_lengths, letters), reverse=True)
p_s, cl_s, l_s = zip(*sorted_data)

x_pos = np.arange(len(letters))
color_map = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(letters)))
bars1 = ax.bar(x_pos, cl_s, color=color_map, alpha=0.8)
ax2 = ax.twinx()
ax2.plot(x_pos, p_s, 'o-', color='steelblue', linewidth=2, markersize=5, label='Probability')
ax.axhline(H, color='red', linestyle='--', linewidth=1.5, label=f'Entropy H={H:.2f}')
ax.axhline(avg_length, color='darkgreen', linestyle=':', linewidth=1.5, label=f'Avg Huffman={avg_length:.2f}')
ax.set_xticks(x_pos)
ax.set_xticklabels(l_s, fontsize=8)
ax.set_ylabel('Code length [bits]')
ax2.set_ylabel('Probability', color='steelblue')
ax.set_title('Huffman codes: high-frequency\nletters get shorter codes')
ax.legend(fontsize=8, loc='upper left')

ax = axes[1]
compression_ratio = [fixed_length / len(codes[l]) for l in letters]
sorted_by_freq = sorted(zip(p_sorted, letters), reverse=True)
p_rank, l_rank = zip(*sorted_by_freq)
cr_rank = [fixed_length / len(codes[l]) for l in l_rank]
ax.bar(range(len(l_rank)), cr_rank, 
       color=['seagreen' if cr > 1 else 'coral' for cr in cr_rank], alpha=0.8)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='No compression')
ax.set_xticks(range(len(l_rank)))
ax.set_xticklabels(l_rank, fontsize=8)
ax.set_ylabel('Compression ratio (fixed ÷ Huffman)')
ax.set_title('Compression ratio per letter\n(sorted by frequency)')
ax.legend()

plt.suptitle('Section 10: Huffman Coding and Source Compression', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 11. Channel Capacity & Noisy Channel Theorem

### The Communication Channel

A **discrete memoryless channel (DMC)** is characterized by:
- Input alphabet `X`
- Output alphabet `Y`  
- Transition probabilities `p(y|x)` — the conditional probability of receiving `y` given `x` was sent

### Channel Capacity

The **channel capacity** is the maximum mutual information between input and output, optimized over all input distributions:

$$\boxed{C = \max_{p(x)} I(X; Y) \quad [\text{bits per channel use}]}$$

### Shannon's Noisy Channel Coding Theorem

> **Theorem:** For a channel with capacity `C` bits/use:
> - If the transmission rate `R < C`, there exist codes that achieve arbitrarily low error probability.
> - If `R > C`, error probability is bounded away from zero no matter what code is used.

This is arguably the most profound result in information theory. It says *reliable communication is always possible* as long as you stay below capacity — regardless of how noisy the channel is.

### The Binary Symmetric Channel (BSC)

A BSC flips each bit independently with probability `p`:
- Capacity: `C = 1 - H(p) = 1 - H_b(p)` bits/use
- At `p = 0`: perfect channel, `C = 1` bit
- At `p = 0.5`: completely random, `C = 0` bits

### The Gaussian Channel (AWGN)

For a channel with additive white Gaussian noise and signal-to-noise ratio `SNR`:

$$C = \frac{1}{2} \log_2(1 + \text{SNR}) \quad \text{[Shannon–Hartley theorem]}$$

This formula underlies WiFi, 5G, and all digital communications.

In [ ]:
# ============================================================
# SECTION 11: Channel Capacity
# ============================================================

def binary_entropy(p):
    """H_b(p) = -p log2(p) - (1-p) log2(1-p)"""
    if p <= 0 or p >= 1:
        return 0
    return -p * np.log2(p) - (1-p) * np.log2(1-p)

def bsc_capacity(p):
    """Capacity of Binary Symmetric Channel with flip probability p."""
    return 1 - binary_entropy(p)

def awgn_capacity(snr_linear):
    """Shannon-Hartley capacity of AWGN channel."""
    return 0.5 * np.log2(1 + snr_linear)


fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: BSC capacity
ax = axes[0]
p_flip = np.linspace(0, 1, 500)
capacities = [bsc_capacity(p) for p in p_flip]
ax.plot(p_flip, capacities, 'steelblue', linewidth=2.5)
ax.fill_between(p_flip, capacities, alpha=0.15, color='steelblue')
ax.axvline(0.5, color='coral', linestyle='--', alpha=0.7, label='p=0.5: C=0 (useless)')
ax.axvline(0.0, color='seagreen', linestyle='--', alpha=0.7, label='p=0: C=1 (perfect)')
ax.scatter([0.1, 0.3], [bsc_capacity(0.1), bsc_capacity(0.3)],
           color='coral', s=80, zorder=5)
for p in [0.1, 0.3]:
    ax.annotate(f'p={p}\nC={bsc_capacity(p):.2f}b',
                xy=(p, bsc_capacity(p)), xytext=(p+0.05, bsc_capacity(p)+0.05), fontsize=8)
ax.set_xlabel('Flip probability p')
ax.set_ylabel('Channel capacity [bits/use]')
ax.set_title('Binary Symmetric Channel (BSC)\nCapacity vs noise')
ax.legend(fontsize=9)

# Center: AWGN Shannon-Hartley
ax = axes[1]
snr_db  = np.linspace(-10, 40, 500)
snr_lin = 10 ** (snr_db / 10)
cap_awgn = [awgn_capacity(s) for s in snr_lin]
ax.plot(snr_db, cap_awgn, 'coral', linewidth=2.5)
# Annotate real-world standards
standards = [
    ('WiFi 802.11b\n(11 Mbps)', 10, None),
    ('WiFi 802.11ac\n(3.5 Gbps)', 30, None),
]
for label, snr, _ in standards:
    c = awgn_capacity(10**(snr/10))
    ax.scatter([snr], [c], color='steelblue', s=80, zorder=5)
    ax.annotate(label, xy=(snr, c), xytext=(snr-8, c+0.5), fontsize=8,
                arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_xlabel('SNR [dB]')
ax.set_ylabel('Capacity [bits/channel use]')
ax.set_title('AWGN Channel (Shannon-Hartley)\nC = ½ log₂(1 + SNR)')

# Right: Capacity region concept (2-user MAC)
ax = axes[2]
# Simplified illustration of capacity region
R1 = np.linspace(0, 2, 200)
C1, C2, C12 = 2.0, 1.5, 3.0
# Rate region constraints: R1 ≤ C1, R2 ≤ C2, R1+R2 ≤ C12
R2_c1   = np.minimum(C2, C12 - R1)     # R1+R2 ≤ C12
R2_c1   = np.clip(R2_c1, 0, C2)
mask = R1 <= C1
ax.fill_between(R1[mask], R2_c1[mask], alpha=0.25, color='steelblue', label='Capacity region')
ax.plot(R1[mask], R2_c1[mask], 'steelblue', linewidth=2)
ax.axvline(C1, color='gray', linestyle=':', alpha=0.5)
ax.axhline(C2, color='gray', linestyle=':', alpha=0.5)
ax.annotate('Sum capacity\nR₁+R₂ ≤ C₁₂', xy=(1.2, 1.6), fontsize=9, color='steelblue')
ax.set_xlabel('Rate R₁ [bits/use]')
ax.set_ylabel('Rate R₂ [bits/use]')
ax.set_title('Capacity region\n(Multiple Access Channel sketch)')
ax.set_xlim(0, 2.5)
ax.set_ylim(0, 2.5)
ax.legend()

plt.suptitle('Section 11: Channel Capacity & Shannon Theorem', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 12. Information Theory in Machine Learning

### 12.1 Decision Trees and Information Gain

When building a decision tree, we choose the feature `A` that maximizes the **information gain** — the reduction in entropy about the label `Y`:

$$\text{IG}(Y; A) = H(Y) - H(Y|A) = I(Y; A)$$

This is simply the *mutual information* between the feature `A` and the label `Y`!

**ID3 algorithm:** Greedily pick the feature maximizing `IG` at each node.

### 12.2 Maximum Likelihood Estimation = Cross-Entropy Minimization

Given data `{x_i}` drawn from true distribution `P`, and a model `Q_θ`:

$$\hat{\theta}_{\text{MLE}} = \arg\max_\theta \sum_i \log q_\theta(x_i) = \arg\min_\theta H(P_{\text{data}}, Q_\theta)$$

MLE is exactly cross-entropy minimization. This unifies two seemingly different frameworks.

### 12.3 Entropy Regularization in RL

In **maximum entropy RL** (Soft Actor-Critic, etc.), we add an entropy bonus to the reward:

$$J(\pi) = \sum_t \mathbb{E}[r_t + \alpha H(\pi(\cdot|s_t))]$$

This encourages *exploration* by rewarding stochastic policies. The temperature `α` controls the exploration-exploitation trade-off.

### 12.4 Naive Bayes Classifier

The Naive Bayes classifier is a direct application of Bayes' theorem — it models the joint probability `P(Y, X)` and makes predictions by computing conditional probabilities. The "naive" assumption of feature independence is an entropy-minimizing factorization of the joint distribution.

In [ ]:
# ============================================================
# SECTION 12: Information Theory in Machine Learning
# ============================================================

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# --- 12.1 Information Gain in Decision Trees ---
print("=== 12.1 Information Gain for Decision Tree Splitting ===")

iris = load_iris()
X, y = iris.data, iris.target

def information_gain(X_col, y, threshold):
    """IG of splitting feature X_col at threshold for labels y."""
    # Parent entropy
    counts = np.bincount(y)
    p_parent = counts / counts.sum()
    H_parent = entropy(p_parent)
    
    # Split
    left_mask  = X_col <= threshold
    right_mask = ~left_mask
    n = len(y)
    
    def child_entropy(mask):
        if mask.sum() == 0:
            return 0
        c = np.bincount(y[mask], minlength=len(counts))
        p = c / c.sum()
        return entropy(p)
    
    H_left  = child_entropy(left_mask)
    H_right = child_entropy(right_mask)
    H_children = (left_mask.sum() * H_left + right_mask.sum() * H_right) / n
    
    return H_parent - H_children

# Find best split for petal length (most informative feature)
feature_idx = 2  # petal length
thresholds = np.linspace(X[:, feature_idx].min(), X[:, feature_idx].max(), 100)
ig_scores = [information_gain(X[:, feature_idx], y, t) for t in thresholds]

best_t = thresholds[np.argmax(ig_scores)]
best_ig = max(ig_scores)
print(f"Best split: petal length ≤ {best_t:.2f} cm")
print(f"Information gain: {best_ig:.4f} bits")

# Compare all features
print("\nMax information gain per feature:")
for i, feat in enumerate(iris.feature_names):
    thresholds_f = np.linspace(X[:,i].min(), X[:,i].max(), 100)
    ig_f = max(information_gain(X[:,i], y, t) for t in thresholds_f)
    print(f"  {feat}: {ig_f:.4f} bits")

# --- Visual ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.plot(thresholds, ig_scores, 'steelblue', linewidth=2)
ax.axvline(best_t, color='coral', linestyle='--', linewidth=1.5,
           label=f'Best split = {best_t:.2f}\nIG = {best_ig:.3f} bits')
ax.fill_between(thresholds, ig_scores, alpha=0.1, color='steelblue')
ax.set_xlabel('Petal length threshold [cm]')
ax.set_ylabel('Information Gain [bits]')
ax.set_title('Information gain at each\npossible split threshold')
ax.legend(fontsize=9)

# Feature comparison bar chart
ax = axes[1]
feat_ig = []
for i in range(4):
    thresholds_f = np.linspace(X[:,i].min(), X[:,i].max(), 100)
    ig_f = max(information_gain(X[:,i], y, t) for t in thresholds_f)
    feat_ig.append(ig_f)
colors_f = ['coral' if v == max(feat_ig) else 'steelblue' for v in feat_ig]
bars = ax.bar(['Sepal\nlength','Sepal\nwidth','Petal\nlength','Petal\nwidth'],
               feat_ig, color=colors_f, alpha=0.85)
for bar, val in zip(bars, feat_ig):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.3f}b',
            ha='center', fontsize=9)
ax.set_ylabel('Max Information Gain [bits]')
ax.set_title('Best feature to split on\n(first tree node)')

# Decision tree entropy visualization
ax = axes[2]
dt = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
dt.fit(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)
acc = accuracy_score(y_test, y_pred)

# Depth vs accuracy
depths = range(1, 11)
train_accs, test_accs = [], []
for d in depths:
    dt_d = DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=42)
    dt_d.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, dt_d.predict(X_train)))
    test_accs.append(accuracy_score(y_test, dt_d.predict(X_test)))

ax.plot(depths, train_accs, 'o-', color='steelblue', linewidth=2, label='Train accuracy')
ax.plot(depths, test_accs,  's-', color='coral',     linewidth=2, label='Test accuracy')
ax.set_xlabel('Tree depth')
ax.set_ylabel('Accuracy')
ax.set_title('Decision tree (entropy criterion)\naccuracy vs depth')
ax.legend()
ax.set_ylim(0.8, 1.02)

plt.suptitle('Section 12: Information Gain in Decision Trees', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 13. Information Theory in Deep Learning

### 13.1 Cross-Entropy Loss: The Standard

The ubiquitous classification loss in neural networks is the cross-entropy:

$$\mathcal{L}_{\text{CE}} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{k=1}^{K} y_{ik} \log \hat{y}_{ik}$$

where `y_{ik}` is the one-hot true label and `ŷ_{ik} = softmax(z_{ik})`.

### 13.2 Variational Autoencoders (VAE)

The VAE trains by maximizing the **Evidence Lower BOund (ELBO)**:

$$\mathcal{L}_{\text{ELBO}} = \underbrace{\mathbb{E}_{z \sim q}[\log p(x|z)]}_{\text{reconstruction}} - \underbrace{D_{\text{KL}}(q(z|x) \| p(z))}_{\text{regularization}}$$

- The **reconstruction term** maximizes the likelihood of the data given the latent code
- The **KL term** ensures the learned posterior `q(z|x)` stays close to the prior `p(z) = N(0, I)`

### 13.3 Knowledge Distillation

In knowledge distillation (Hinton et al., 2015), we train a student model to match the *soft targets* of a teacher:

$$\mathcal{L}_{\text{KD}} = \alpha \cdot \mathcal{L}_{\text{CE}}(y, \hat{y}_{\text{student}}) + (1-\alpha) \cdot \tau^2 \cdot D_{\text{KL}}(p_{\text{teacher}}^\tau \| p_{\text{student}}^\tau)$$

where `τ` is the temperature parameter that softens probability distributions.

### 13.4 Perplexity in Language Models

**Perplexity** measures how well a language model predicts text. For a test corpus of `N` tokens:

$$\text{PPL} = 2^{H(P, Q)} = 2^{-\frac{1}{N} \sum_{i} \log_2 P_{\text{model}}(w_i | w_{<i})}$$

GPT-2 had PPL ~18 on Penn Treebank. GPT-4 class models achieve PPL <10. Human-level is approximately 2–3.

In [ ]:
# ============================================================
# SECTION 13: Information Theory in Deep Learning
# ============================================================

# --- 13.1 Cross-entropy loss: temperature and softmax ---
print("=== 13.1 Temperature scaling and cross-entropy ===")

def softmax(logits, temperature=1.0):
    """Softmax with temperature parameter."""
    logits = np.array(logits, dtype=float)
    logits = logits / temperature
    logits -= logits.max()  # numerical stability
    exp_l = np.exp(logits)
    return exp_l / exp_l.sum()

# Raw logits from a classifier
logits = np.array([3.0, 1.5, 0.5, -0.5, -1.0])
classes = ['Cat', 'Dog', 'Bird', 'Fish', 'Snake']

print("\nSoftmax outputs at different temperatures:")
print(f"{'Class':8s}  {'τ=0.1':8s}  {'τ=0.5':8s}  {'τ=1.0':8s}  {'τ=2.0':8s}  {'τ=10.0':8s}")
for i, cls in enumerate(classes):
    row = [softmax(logits, τ)[i] for τ in [0.1, 0.5, 1.0, 2.0, 10.0]]
    print(f"{cls:8s}  " + "  ".join(f"{v:.4f}" for v in row))


# --- VAE KL divergence (analytical for Gaussian) ---
print("\n=== 13.2 VAE KL term (reparameterization) ===")

def vae_kl_loss(mu, log_var):
    """KL(N(mu, exp(log_var)) || N(0,1)) per dimension.
    Analytical formula: -0.5 * (1 + log_var - mu^2 - exp(log_var))
    """
    return -0.5 * (1 + log_var - mu**2 - np.exp(log_var))

# Show how KL varies with mu and log_var
mu_vals   = np.linspace(-3, 3, 50)
lv_vals   = np.linspace(-3, 1, 50)
MU, LV    = np.meshgrid(mu_vals, lv_vals)
KL_grid   = vae_kl_loss(MU, LV)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: Temperature softmax visualization
ax = axes[0]
temps = [0.1, 0.5, 1.0, 2.0, 10.0]
x_pos = np.arange(len(classes))
for τ, color, lw in zip(temps, plt.cm.RdYlGn(np.linspace(0.9,0.1,5)), [2.5]*5):
    ax.plot(x_pos, softmax(logits, τ), 'o-', linewidth=lw, label=f'τ={τ}', alpha=0.9)
ax.set_xticks(x_pos)
ax.set_xticklabels(classes)
ax.set_ylabel('Softmax probability')
ax.set_title('Temperature τ controls\nprobability sharpness')
ax.legend(fontsize=8)

# Center: VAE KL heatmap
ax = axes[1]
im = ax.contourf(mu_vals, lv_vals, np.clip(KL_grid, 0, 5), levels=30, cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='KL divergence (nats)')
ax.contour(mu_vals, lv_vals, KL_grid, levels=[0], colors='blue', linewidths=2)
ax.scatter([0], [0], color='blue', s=150, zorder=5, label='KL=0 (mu=0, logvar=0)')
ax.set_xlabel('Mean μ')
ax.set_ylabel('Log variance log(σ²)')
ax.set_title('VAE KL term: KL(N(μ,σ²) ‖ N(0,1))\nblue = zero KL')
ax.legend(fontsize=9)

# Right: Perplexity comparison
ax = axes[2]
# Simulated perplexity of different language models
models = ['GPT-1\n(2018)', 'GPT-2\n(2019)', 'GPT-3\n(2020)', 'ChatGPT\nClass\n(2022)', 
          'GPT-4\nClass\n(2023)', 'Human\nlevel']
ppl_penn = [35, 18, 20, 10, 7, 2.5]  # approximate Penn Treebank perplexities
# These are representative/approximate numbers for illustration

colors_ppl = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(models)))
bars = ax.bar(models, ppl_penn, color=colors_ppl, alpha=0.85)
for bar, val in zip(bars, ppl_penn):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.2, f'PPL={val}',
            ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Perplexity (lower = better)')
ax.set_title('Language model perplexity\n(approximate, illustrative)')
ax.set_yscale('log')

plt.suptitle('Section 13: Information Theory in Deep Learning', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Cross-entropy loss demonstration
print("\n=== Cross-entropy loss for different prediction qualities ===")
y_true = np.array([0, 0, 1, 0])  # one-hot: class 2 is correct
for name, y_pred in [
    ('Confident correct', [0.01, 0.01, 0.97, 0.01]),
    ('Uncertain',         [0.25, 0.25, 0.25, 0.25]),
    ('Confident WRONG',  [0.97, 0.01, 0.01, 0.01]),
]:
    ce = cross_entropy(y_true, y_pred)
    print(f"  {name:25s}: CE = {ce:.4f} bits")

---
## 14. Information Bottleneck Theory

### The Fundamental Idea

The **Information Bottleneck (IB)** method (Tishby, Pereira, Bialek, 1999) provides a theoretical framework for understanding what neural networks are doing when they learn representations.

Given input `X`, target `Y`, and latent representation `T` (the bottleneck), the IB objective is:

$$\min_{p(t|x)} \underbrace{I(X; T)}_{\text{compress}} - \beta \cdot \underbrace{I(T; Y)}_{\text{preserve label info}}$$

- **Compress the input:** Minimize `I(X; T)` — throw away irrelevant information in `X`
- **Preserve relevant information:** Maximize `I(T; Y)` — keep what predicts the label
- **`β`** controls the compression-accuracy trade-off (like temperature)

### The IB Curve and the Data Processing Inequality

By the **Data Processing Inequality (DPI)**: since `Y → X → T` forms a Markov chain:
$$I(T; Y) \leq I(X; Y)$$

Processing can only lose information. The IB curve traces the **optimal** trade-off between compression and prediction.

### Neural Networks as Information Bottlenecks (Tishby & Schwartz-Ziv, 2017)

The controversial but influential claim: during training, neural network layers first **memorize** (increasing `I(T; X)`) then **compress** (decreasing `I(T; X)` while maintaining `I(T; Y)`).

This is the *two-phase* learning hypothesis:
1. **Fitting phase:** SGD fits the data, all layers increase MI with `X`
2. **Compression phase:** Layers compress, discarding irrelevant information about `X`

*(Note: this hypothesis is debated in the field — it depends on activation functions and network architecture.)*

In [ ]:
# ============================================================
# SECTION 14: Information Bottleneck
# ============================================================

# --- Illustrate the IB trade-off curve ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: The IB Curve (theoretical)
ax = axes[0]
# Parametric IB curve: as beta increases, more compression
I_TX = np.array([4.0, 3.5, 3.0, 2.5, 2.0, 1.8, 1.5, 1.2, 0.8, 0.5, 0.2, 0.0])
I_TY = np.array([0.0, 0.3, 0.8, 1.4, 1.8, 1.9, 1.95, 1.97, 1.98, 1.9, 1.5, 0.0])
# I(X;Y) = 2.0 bits (total information in X about Y)
IXY  = 2.0

ax.plot(I_TX, I_TY, 'steelblue', linewidth=2.5, label='IB optimal curve')
ax.fill_between(I_TX, I_TY, alpha=0.1, color='steelblue')
ax.axhline(IXY, color='gray', linestyle='--', alpha=0.5, label=f'I(X;Y) = {IXY} bits (ceiling)')

# Mark different beta regimes
for i, (itx, ity, label) in enumerate([
    (3.5, 0.3,  'High β\n(over-compress)'),
    (2.0, 1.8,  'Optimal β\n(good repr.)'),
    (0.5, 1.9,  'Low β\n(minimal compression)'),
]):
    ax.scatter([itx], [ity], s=100, zorder=5, color=['coral','seagreen','purple'][i])
    ax.annotate(label, xy=(itx, ity), xytext=(itx+0.2, ity-0.2), fontsize=8,
                color=['coral','seagreen','purple'][i])

ax.set_xlabel('I(T; X) [bits]  — compression')
ax.set_ylabel('I(T; Y) [bits]  — predictive power')
ax.set_title('Information Bottleneck curve\n(trade-off: compression vs accuracy)')
ax.legend(fontsize=9)

# Center: Data Processing Inequality visualization
ax = axes[1]
# Markov chain: Y → X → T1 → T2 → T3
# Mutual information can only decrease along the chain
chain_labels = ['I(Y;X)\n(original)', 'I(Y;T₁)\nlayer 1', 'I(Y;T₂)\nlayer 2', 
                'I(Y;T₃)\nlayer 3', 'I(Y;Ŷ)\noutput']
# Example values showing both compression phases
mi_fitting  = [2.0, 1.95, 1.92, 1.90, 1.88]  # during fitting: high MI maintained
mi_compress = [2.0, 1.98, 1.97, 1.95, 1.93]  # after compression: efficient repr.

x_pos = np.arange(5)
ax.plot(x_pos, mi_fitting,  'o-', color='coral',    linewidth=2, label='During fitting', markersize=8)
ax.plot(x_pos, mi_compress, 's--', color='steelblue', linewidth=2, label='After compression', markersize=8)
ax.fill_between(x_pos, mi_fitting, mi_compress, alpha=0.15, color='purple', label='Compression gain')
ax.set_xticks(x_pos)
ax.set_xticklabels(chain_labels, fontsize=8)
ax.set_ylabel('I(T; Y) [bits]')
ax.set_title('Data Processing Inequality\nin a neural network')
ax.legend(fontsize=9)
ax.set_ylim(1.8, 2.05)

# Right: IB in practice — varying beta
ax = axes[2]
beta_vals = np.logspace(-2, 2, 200)
# Simplified model: as beta grows, we compress more
# (using a toy parametric model for illustration)
C = IXY  # max predictive info
I_TX_beta = C * (1 + 1/beta_vals) * np.exp(-beta_vals * 0.3)  # toy model
I_TX_beta = np.clip(I_TX_beta, 0, 4)
I_TY_beta = C * (1 - np.exp(-beta_vals * 0.5))  # approaches C(X;Y)
I_TY_beta = np.clip(I_TY_beta, 0, C)

ax.semilogx(beta_vals, I_TX_beta, 'coral', linewidth=2, label='I(T;X) — complexity')
ax.semilogx(beta_vals, I_TY_beta, 'steelblue', linewidth=2, label='I(T;Y) — accuracy')
ax.axhline(IXY, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('β (compression strength)')
ax.set_ylabel('[bits]')
ax.set_title('Effect of β on compression\nvs predictive information')
ax.legend(fontsize=9)

plt.suptitle('Section 14: Information Bottleneck Theory', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Information Bottleneck in modern AI:")
print("  VAE:            β-VAE is literally the IB objective with a neural encoder!")
print("  Transformers:   Attention heads learn to compress context into relevant queries")
print("  RLHF:           KL penalty to reference model is an information bottleneck")
print("  Model pruning:  Removing weights that don't carry predictive information")

---
## 15. Minimum Description Length & Bayesian Inference

### 15.1 Kolmogorov Complexity

The **Kolmogorov complexity** `K(x)` of a string `x` is the length of the shortest program that outputs `x`. It is the ultimate measure of information — but it is *incomputable*.

Shannon entropy can be viewed as the *expected* Kolmogorov complexity under a distribution.

### 15.2 Minimum Description Length (MDL)

**Rissanen's MDL principle (1978):** The best model is the one that allows the *shortest description* of the data. It formalizes Occam's Razor in information-theoretic terms:

$$\text{MDL}(H, D) = \underbrace{L(D|H)}_{\text{data given model}} + \underbrace{L(H)}_{\text{model complexity}}$$

- `L(D|H)` = bits needed to describe data given the hypothesis (related to negative log-likelihood)
- `L(H)` = bits needed to describe the model itself (regularization/prior)

### 15.3 Bayesian Connection

MDL is equivalent to MAP Bayesian inference:

$$\arg\min_H \left[ -\log P(D|H) - \log P(H) \right] = \arg\max_H P(H|D)$$

The prior `P(H)` encodes model complexity; simpler models have higher priors.

### 15.4 AI Applications

| Concept | MDL Interpretation |
|---------|-------------------|
| L1/L2 regularization | Penalize model description length |
| Dropout | Stochastic description of weights |
| BIC/AIC | Approximate MDL for model selection |
| Variational inference | Approximate the minimum description length posterior |
| Weight quantization | Reduce bits needed to describe weights |

In [ ]:
# ============================================================
# SECTION 15: MDL, Regularization, and Bayesian Connections
# ============================================================

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

# --- MDL as model selection: polynomial degree fitting ---
np.random.seed(42)
n = 40
X_raw = np.linspace(-3, 3, n)
y_true = np.sin(X_raw)  # true function
y_noisy = y_true + np.random.normal(0, 0.3, n)

X_feat = X_raw.reshape(-1, 1)
X_plot = np.linspace(-3.5, 3.5, 200).reshape(-1, 1)

degrees = [1, 3, 7, 15]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, degree in zip(axes.flatten(), degrees):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=degree)),
        ('reg', LinearRegression())
    ])
    pipe.fit(X_feat, y_noisy)
    y_pred_plot = pipe.predict(X_plot)
    y_pred_train = pipe.predict(X_feat)
    
    mse = mean_squared_error(y_noisy, y_pred_train)
    
    # MDL: description_length ≈ -log P(data|model) + k/2 * log(n) [BIC approximation]
    k = degree + 1   # number of parameters
    n_fit = len(y_noisy)
    sigma2_hat = mse
    log_likelihood = -n_fit/2 * (np.log(2*np.pi*sigma2_hat) + 1)
    bic = -2 * log_likelihood + k * np.log(n_fit)  # BIC ≈ MDL
    
    ax.scatter(X_raw, y_noisy, color='gray', s=20, alpha=0.7, label='Noisy data')
    ax.plot(X_plot, np.sin(X_plot), 'steelblue', linewidth=1.5, alpha=0.4, label='True: sin(x)')
    ax.plot(X_plot, y_pred_plot, 'coral', linewidth=2.5, label=f'Degree {degree}')
    ax.set_ylim(-2.5, 2.5)
    ax.set_xlim(-3.5, 3.5)
    ax.set_title(f'Polynomial degree {degree}\nMSE={mse:.3f}  |  BIC≈MDL={bic:.1f}  |  k={k} params')
    ax.legend(fontsize=8)
    if bic == min(
        -2*(-n_fit/2*(np.log(2*np.pi*mean_squared_error(y_noisy,
            Pipeline([('p', PolynomialFeatures(d)), ('r', LinearRegression())]).fit(X_feat,y_noisy).predict(X_feat))+1e-10)+1)) 
        + (d+1)*np.log(n_fit)
        for d in degrees
    ):
        ax.set_facecolor('#f0fff0')

plt.suptitle('Section 15: MDL as Model Selection (BIC ≈ MDL)\nGreen background = lowest BIC (preferred by MDL)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Compute BIC for all degrees
print("\nBIC (≈ MDL) for polynomial degrees:")
print(f"{'Degree':8s} {'Params':8s} {'MSE':10s} {'Log-lik':12s} {'BIC':10s}")
bics = []
for degree in range(1, 16):
    pipe = Pipeline([('poly', PolynomialFeatures(degree=degree)), ('reg', LinearRegression())])
    pipe.fit(X_feat, y_noisy)
    mse = mean_squared_error(y_noisy, pipe.predict(X_feat))
    k = degree + 1
    ll = -n/2 * (np.log(2*np.pi*max(mse, 1e-10)) + 1)
    bic = -2*ll + k*np.log(n)
    bics.append(bic)
    marker = ' ← BEST' if bic == min(bics) else ''
    if degree <= 5 or degree in [7, 10, 15]:
        print(f"  {degree:6d}   {k:6d}   {mse:8.4f}   {ll:10.2f}   {bic:8.2f}{marker}")

---
## Summary: Information Theory Cheat Sheet

| Concept | Formula | Unit | Use in AI |
|---------|---------|------|----------|
| Self-information | `-log₂ p(x)` | bits | Measuring event surprise |
| Shannon Entropy | `-Σ p(x) log₂ p(x)` | bits | Uncertainty measure, compression bound |
| Conditional Entropy | `H(Y|X) = H(X,Y) - H(X)` | bits | Remaining uncertainty after observation |
| Mutual Information | `H(X) + H(Y) - H(X,Y)` | bits | Feature selection, IB theory, correlation |
| KL Divergence | `Σ p log(p/q)` | nats/bits | VAE, PPO, distillation, MLE |
| Cross-Entropy | `H(P) + D_KL(P‖Q)` | bits | Neural network loss function |
| Jensen-Shannon | `½KL(P‖M) + ½KL(Q‖M)` | bits | GANs, symmetric divergence |
| Perplexity | `2^H(P,Q)` | - | LLM evaluation |
| Channel Capacity | `max I(X;Y)` | bits/use | Theoretical communication limit |
| Information Gain | `H(Y) - H(Y|A)` | bits | Decision tree splitting |
| MDL/BIC | `-log P(D|H) - log P(H)` | bits | Model selection, regularization |

### Key Inequalities

```
0 ≤ H(X) ≤ log₂|X|            (entropy bounds)
H(Y|X) ≤ H(Y)                  (conditioning reduces entropy)
I(X;Y) ≥ 0                     (MI is non-negative)
I(X;Z) ≤ I(X;Y)                (data processing inequality)
D_KL(P‖Q) ≥ 0                  (Gibbs' inequality)
H(P,Q) ≥ H(P)                  (cross-entropy ≥ entropy)
```

In [ ]:
# ============================================================
# FINAL: Complete information-theoretic toolkit
# ============================================================

print("="*60)
print("INFORMATION THEORY TOOLKIT — QUICK REFERENCE")
print("="*60)

# All core functions in one place
import numpy as np

def H(p):
    """Shannon entropy. p: array of probabilities."""
    p = np.array(p, dtype=float)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

def I_self(prob):
    """Self-information of event with given probability."""
    return -np.log2(prob)

def CE(p, q):
    """Cross-entropy H(P, Q)."""
    p, q = np.array(p), np.array(q)
    mask = p > 0
    return -np.sum(p[mask] * np.log2(q[mask]))

def KL(p, q):
    """KL divergence D_KL(P || Q)."""
    return CE(p, q) - H(p)

def JSD(p, q):
    """Jensen-Shannon divergence."""
    m = 0.5 * (np.array(p) + np.array(q))
    return 0.5 * KL(p, m) + 0.5 * KL(q, m)

def MI(joint):
    """Mutual information I(X;Y) from joint distribution matrix."""
    J = np.array(joint)
    return H(J.sum(0)) + H(J.sum(1)) - H(J.flatten())

def PPL(p, q):
    """Perplexity = 2^H(P,Q)."""
    return 2 ** CE(p, q)

# Demo
P = [0.5, 0.3, 0.15, 0.05]
Q = [0.25, 0.25, 0.25, 0.25]

print(f"\nP = {P}")
print(f"Q = {Q} (uniform)")
print(f"\nH(P)      = {H(P):.4f} bits")
print(f"H(Q)      = {H(Q):.4f} bits")
print(f"CE(P,Q)   = {CE(P,Q):.4f} bits")
print(f"KL(P||Q)  = {KL(P,Q):.4f} bits")
print(f"KL(Q||P)  = {KL(Q,P):.4f} bits  (asymmetric!)")
print(f"JSD(P,Q)  = {JSD(P,Q):.4f} bits  (symmetric)")
print(f"PPL(P,Q)  = {PPL(P,Q):.4f}  (perplexity)")
print(f"\nVerification: CE = H(P) + KL(P||Q)")
print(f"  {H(P):.4f} + {KL(P,Q):.4f} = {H(P)+KL(P,Q):.4f} ≈ {CE(P,Q):.4f} ✓")

---
## 16. Further Reading & Resources

### 📚 Foundational Papers

| Paper | Authors | Year | Link | Why Read |
|-------|---------|------|------|----------|
| **A Mathematical Theory of Communication** | Claude Shannon | 1948 | [Bell Labs PDF](http://people.math.harvard.edu/~ctm/home/text/others/shannon/entropy/entropy.pdf) | *The original* — foundational and readable |
| **Information Theory: A Tutorial Introduction** | James V. Stone | 2018 | [arXiv:1802.05968](https://arxiv.org/abs/1802.05968) | Best modern intro — free on arXiv |
| **A Brief Tutorial on Information Theory** | Tohme & Bialek | 2024 | [arXiv:2402.16556](https://arxiv.org/abs/2402.16556) | Physics perspective, very accessible |
| **Information-Theoretic Foundations of ML** | Various | 2024 | [ISIT 2024 Tutorials](https://2024.ieee-isit.org/tutorials) | Current research directions |

### 📖 Books (From Beginner to Advanced)

| Book | Authors | Level | Notes |
|------|---------|-------|-------|
| **An Introduction to Information Theory** | John Pierce | Beginner | Non-mathematical, great intuition builder |
| **Information Theory: A Tutorial Introduction** | James V. Stone | Beginner-Intermediate | Best single book to start with |
| **Elements of Information Theory** | Cover & Thomas | Intermediate | *The standard graduate textbook* |
| **Information Theory, Inference & Learning Algorithms** | David MacKay | Intermediate | [Free PDF](http://www.inference.org.uk/mackay/itila/) — connects IT, ML and Bayesian inference |
| **Information Theory and Evolution** | John Avery | Intermediate | Connections to physics and biology |
| **The Information** | James Gleick | Popular | Excellent non-technical history |
| **A Mind at Play** | Soni & Goodman | Popular | Biography of Claude Shannon |

### 🎥 Video Lectures

| Resource | Link | Level |
|----------|------|-------|
| **Stanford CS229: Machine Learning** (IT sections) | [cs229.stanford.edu](https://cs229.stanford.edu) | Intermediate |
| **MIT 6.441: Information Theory** | [OCW MIT](https://ocw.mit.edu) | Advanced |
| **3Blue1Brown: Information Entropy** | [YouTube](https://www.youtube.com/watch?v=v68zYyaEmEA) | Beginner |
| **IEEE IT Society Video Tutorials** | [itsoc.org/resources](https://www.itsoc.org/resources/resources-on-information-theory) | Various |

### 🔬 AI/ML Specific Papers

| Paper | Topic | Link |
|-------|-------|------|
| Tishby et al. (1999) | Original Information Bottleneck | [arXiv:physics/0004057](https://arxiv.org/abs/physics/0004057) |
| Tishby & Schwartz-Ziv (2017) | Opening the Black Box of DNNs via IT | [arXiv:1703.00810](https://arxiv.org/abs/1703.00810) |
| Kingma & Welling (2014) | VAE — KL in deep generative models | [arXiv:1312.6114](https://arxiv.org/abs/1312.6114) |
| Goodfellow et al. (2014) | GANs — JSD connection | [arXiv:1406.2661](https://arxiv.org/abs/1406.2661) |
| Hinton et al. (2015) | Knowledge Distillation | [arXiv:1503.02531](https://arxiv.org/abs/1503.02531) |
| Haarnoja et al. (2018) | SAC — maximum entropy RL | [arXiv:1801.01290](https://arxiv.org/abs/1801.01290) |

### 🌐 Online Resources

| Resource | URL | Notes |
|----------|-----|-------|
| **MacKay's Book (Free PDF)** | http://www.inference.org.uk/mackay/itila/ | Complete textbook free online |
| **Stanford Info Theory Notes** | https://web.stanford.edu/~montanar/RESEARCH/BOOK/partA.pdf | Rigorous graduate notes |
| **MIT LIDS IT Book** | https://people.lids.mit.edu/yp/homepage/data/itbook-export.pdf | Modern ML-focused IT textbook |
| **IEEE IT Society** | https://www.itsoc.org/resources | Journals, surveys, video tutorials |
| **Towards Data Science** | https://towardsdatascience.com | Applied articles (variable quality) |

---

### Learning Path Recommendation

```
BEGINNER PATH (2-4 weeks)
  1. Watch 3Blue1Brown entropy video  [30 min]
  2. Read Stone's Tutorial arXiv paper [2-3 hours]
  3. Work through this notebook       [3-4 hours]
  4. Pierce: An Introduction to IT    [1-2 weeks]

INTERMEDIATE PATH (2-3 months)
  5. MacKay Chapters 1-8 (free PDF)   [3-4 weeks]
  6. Cover & Thomas Chapters 1-5      [4-6 weeks]
  7. Tishby (1999) + Tishby (2017)    [1 week]

ADVANCED / AI-FOCUSED PATH
  8. MIT 6.441 lecture notes          [4-6 weeks]
  9. ISIT 2024 tutorial papers        [ongoing]
  10. IEEE IT Society JSAIT papers    [ongoing]
```